In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Image
import os, datetime
import seaborn as sns

from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
from sklearn.metrics import roc_curve, auc, classification_report, confusion_matrix
from sklearn.cluster import KMeans

import tensorflow as tf
import keras
from tensorflow.keras import layers, models, callbacks
from tensorflow.keras import optimizers
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout
from tensorflow.keras.utils import plot_model

import pickle

import random
random.seed(123)

In [ ]:
file_path = '/content/drive/MyDrive/Colab Notebooks/nids/CICIDS2017-dataset/Monday-WorkingHours.pcap_ISCX.csv'
df = pd.read_csv(file_path)

In [ ]:
df.shape

(529918, 79)

In [ ]:
df.columns = df.columns.str.strip()

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 529918 entries, 0 to 529917
Data columns (total 79 columns):
 #   Column                       Non-Null Count   Dtype  
---  ------                       --------------   -----  
 0   Destination Port             529918 non-null  int64  
 1   Flow Duration                529918 non-null  int64  
 2   Total Fwd Packets            529918 non-null  int64  
 3   Total Backward Packets       529918 non-null  int64  
 4   Total Length of Fwd Packets  529918 non-null  int64  
 5   Total Length of Bwd Packets  529918 non-null  int64  
 6   Fwd Packet Length Max        529918 non-null  int64  
 7   Fwd Packet Length Min        529918 non-null  int64  
 8   Fwd Packet Length Mean       529918 non-null  float64
 9   Fwd Packet Length Std        529918 non-null  float64
 10  Bwd Packet Length Max        529918 non-null  int64  
 11  Bwd Packet Length Min        529918 non-null  int64  
 12  Bwd Packet Length Mean       529918 non-null  float64
 13 

In [ ]:
df.columns

Index(['Destination Port', 'Flow Duration', 'Total Fwd Packets',
       'Total Backward Packets', 'Total Length of Fwd Packets',
       'Total Length of Bwd Packets', 'Fwd Packet Length Max',
       'Fwd Packet Length Min', 'Fwd Packet Length Mean',
       'Fwd Packet Length Std', 'Bwd Packet Length Max',
       'Bwd Packet Length Min', 'Bwd Packet Length Mean',
       'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s',
       'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min',
       'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max',
       'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std',
       'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags',
       'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length',
       'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s',
       'Min Packet Length', 'Max Packet Length', 'Packet Length Mean',
       'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count',
       'SYN Flag Co

In [ ]:
label_counts = df['Label'].value_counts()
print("Distribution of Classes:")
print(label_counts)

normal_count = label_counts.get('BENIGN', 0)
attack_count = label_counts.sum() - normal_count

print(f"\nNormal Connections (BENIGN): {normal_count}")
print(f"Attacks (of all types): {attack_count}")

Distribution of Classes:
Label
BENIGN    529918
Name: count, dtype: int64

Normal Connections (BENIGN): 529918
Attacks (of all types): 0


In [ ]:
features = df.columns.tolist()
features.remove('Label')

train_df = df[df['Label'] == 'BENIGN'].copy()
train_features = train_df[features]

train_features.replace([np.inf, -np.inf], np.nan, inplace=True)

train_features = train_features.fillna(train_features.median())

test_df = df.copy()
test_features = test_df[features]
test_features.replace([np.inf, -np.inf], np.nan, inplace=True)
test_features = test_features.fillna(test_features.median())

print("Train NaNs:", train_features.isna().sum().sum())
print("Train Infs:", np.isinf(train_features.values).sum())
print("Test NaNs:", test_features.isna().sum().sum())
print("Test Infs:", np.isinf(test_features.values).sum())

/tmp/ipython-input-3258822069.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  train_features.replace([np.inf, -np.inf], np.nan, inplace=True)
/tmp/ipython-input-3258822069.py:13: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  test_features.replace([np.inf, -np.inf], np.nan, inplace=True)


Train NaNs: 0
Train Infs: 0
Test NaNs: 0
Test Infs: 0


In [ ]:
scaler = MinMaxScaler()

X_train_scaled = scaler.fit_transform(train_features)
X_test_scaled = scaler.transform(test_features)

print(X_train_scaled.shape, X_test_scaled.shape)

(529918, 78) (529918, 78)


In [ ]:
input_dim = X_train_scaled.shape[1]

# model hyperparameters
batch_size = 512

latent_dim = 32

max_epochs = 10

In [ ]:
tf.version.VERSION

'2.19.0'

In [ ]:
model = tf.keras.models.load_model("/content/drive/MyDrive/Colab Notebooks/nids/models/cnnae/cnnae_base.keras")

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 10 variables whereas the saved optimizer has 18 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [ ]:
@keras.saving.register_keras_serializable()
class Encoder(tf.keras.Model):
    def __init__(self, input_shape, latent_dim, **kwargs):
        super().__init__(**kwargs)
        self.input_shape_ = input_shape
        self.latent_dim = latent_dim

        self.conv1 = layers.Conv1D(32, 3, padding="same", activation="relu")
        self.conv2 = layers.Conv1D(latent_dim, 3, padding="same", activation="relu")
        self.maxpool = layers.MaxPooling1D(2, padding="same")

    def call(self, x):
        x = self.conv1(x)
        x = self.conv2(x)
        return self.maxpool(x)

    def get_config(self):
        return {
            "input_shape": self.input_shape_,
            "latent_dim": self.latent_dim,
        }

In [ ]:
@keras.saving.register_keras_serializable()
class Decoder(tf.keras.Model):
    def __init__(self, latent_dim, **kwargs):
        super().__init__(**kwargs)
        self.latent_dim = latent_dim

        self.deconv1 = layers.Conv1D(latent_dim, 3, padding="same", activation="relu")
        self.upsample = layers.UpSampling1D(2)
        self.output_layer = layers.Conv1D(1, 3, padding="same", activation="linear")

    def call(self, x):
        x = self.deconv1(x)
        x = self.upsample(x)
        return self.output_layer(x)

    def get_config(self):
        return {
            "latent_dim": self.latent_dim,
        }

In [ ]:
@keras.saving.register_keras_serializable()
class ConvAutoencoder(tf.keras.Model):
    def __init__(self, input_shape, latent_dim, **kwargs):
        super().__init__(**kwargs)
        self.input_shape_ = input_shape
        self.latent_dim = latent_dim

        self.encoder = Encoder(input_shape, latent_dim)
        self.decoder = Decoder(latent_dim)

    def call(self, x):
        return self.decoder(self.encoder(x))

    def get_config(self):
        return {
            "input_shape": self.input_shape_,
            "latent_dim": self.latent_dim,
        }

In [ ]:
keras.saving.load_model("cnnae.keras")

<ConvAutoencoder name=conv_autoencoder_6, built=False>

In [ ]:
keras.saving.load_model("/content/drive/MyDrive/Colab Notebooks/nids/models/cnnae/cnnae_base.keras")

/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_lib.py:802: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 10 variables whereas the saved optimizer has 18 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


<ConvAutoencoder name=conv_autoencoder_7, built=True>

In [ ]:
cnnae = ConvAutoencoder((input_dim, 1), 16)
cnnae.compile(optimizer = "adam", loss = "mse")

In [ ]:
X_train_cnn = X_train_scaled.reshape(-1, 78, 1)
X_test_cnn  = X_test_scaled.reshape(-1, 78, 1)

In [ ]:
cnnae.summary()
early_stop = callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history = cnnae.fit(
    X_train_cnn, X_train_cnn,
    epochs=100,
    batch_size=64,
    validation_split=0.1,
    shuffle=True,
    callbacks=[early_stop]
)

X_test_pred = cnnae.predict(X_test_cnn)
mse = np.mean(np.power(X_test_cnn - X_test_pred, 2), axis=1)

X_train_pred = cnnae.predict(X_train_cnn)
mse_train = np.mean(np.power(X_train_cnn - X_train_pred, 2), axis=1)

Model: "conv_autoencoder_9"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ encoder_9 (Encoder)             │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ decoder_9 (Decoder)             │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
7452/7452 ━━━━━━━━━━━━━━━━━━━━ 32s 4ms/step - loss: 0.0043 - val_loss: 0.0017
Epoch 2/100
7452/7452 ━━━━━━━━━━━━━━━━━━━━ 22s 3ms/step - loss: 0.0018 - val_loss: 0.0017
Epoch 3/100
7452/7452 ━━━━━━━━━━━━━━━━━━━━ 21s 3ms/step - loss: 0.0017 - val_loss: 0.0016
Epoch 4/100
7452/7452 ━━━━━━━━━━━━━━━━━━━━ 22s 3ms/step - loss: 0.0017 - val_loss: 0.0016
Epoch 5/100
7452/7452 ━━━━━━━━━━━━━━━━━━━━ 21s 3ms/step - loss: 0.0017 - val_loss: 0.0016
Epoch 6/100
7452/7452 ━━━━━━━━━━━━━━━━━━━━ 22s 3ms/step - loss: 0.0016 - val_loss: 0.0016
Epoch 7/100
7452/7452 ━━━━━━━━━━━━━━━━━━━━ 21s 3ms/step - loss: 0.0016 - val_loss: 0.0016
Epoch 8/100
7452/7452 ━━━━━━━━━━━━━━━━━━━━ 21s 3ms/step - loss: 0.0016 - val_loss: 0.0016
Epoch 9/100
7452/7452 ━━━━━━━━━━━━━━━━━━━━ 22s 3ms/step - loss: 0.0016 - val_loss: 0.0016
Epoch 10/100
7452/7452 ━━━━━━━━━━━━━━━━━━━━ 21s 3ms/step - loss: 0.0016 - val_loss: 0.0016
Epoch 11/100
7452/7452 ━━━━━━━━━━━━━━━━━━━━ 22s 3ms/step - loss: 0.0016 - val_loss: 0.0016
Epoch 12

In [ ]:
cnnae.save("/content/drive/MyDrive/Colab Notebooks/nids/models/cnnae-again/cnnae_base.h5")
cnnae.save("/content/drive/MyDrive/Colab Notebooks/nids/models/cnnae-again/cnnae_base.keras")

In [ ]:
keras.utils.get_custom_objects().update({
    "ConvAutoencoder": ConvAutoencoder,
    "Encoder": Encoder,
    "Decoder": Decoder,
})

In [ ]:
recon = cnnae.predict(X_test_cnn, batch_size=256)
mse = tf.reduce_mean(tf.square(X_test_cnn - recon), axis=[1,2]).numpy()

2070/2070 ━━━━━━━━━━━━━━━━━━━━ 5s 2ms/step


In [ ]:
threshold = np.percentile(mse_train, 95)  # or 97 / 99

In [ ]:
print(threshold)

0.003949724197750069


In [ ]:
print(mse)

[0.00116068 0.00196663 0.00196663 ... 0.00069157 0.00064164 0.00295232]


In [ ]:
np.mean(mse)

np.float64(0.0013106041533240296)